In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("../data/dataset_train.csv")
df_test = pd.read_csv("../data/dataset_test.csv")

df.head()

,session_id,DateTime,user_id,product,campaign_id,webpage_id,product_category_1,product_category_2,user_group_id,gender,age_level,user_depth,city_development_index,var_1,is_click
0,140690,2017-07-02 00:00,858557,C,359520,13787,4,NaN,10.0,Female,4.0,3.0,3.0,0,0
1,333291,2017-07-02 00:00,243253,C,105960,11085,5,NaN,8.0,Female,2.0,2.0,NaN,0,0
2,129781,2017-07-02 00:00,243253,C,359520,13787,4,NaN,8.0,Female,2.0,2.0,NaN,0,0
3,464848,2017-07-02 00:00,1097446,I,359520,13787,3,NaN,3.0,Male,3.0,3.0,2.0,1,0
4,90569,2017-07-02 00:01,663656,C,405490,60305,3,NaN,2.0,Male,2.0,3.0,2.0,1,0


In [3]:
df.columns

Index(['session_id', 'DateTime', 'user_id', 'product', 'campaign_id',
       'webpage_id', 'product_category_1', 'product_category_2',
       'user_group_id', 'gender', 'age_level', 'user_depth',
       'city_development_index', 'var_1', 'is_click'],
      dtype='str')

In [ ]:
df["DateTime"] = pd.to_datetime(df["DateTime"])

dt = df["DateTime"]

df["year"] = dt.dt.year
df["month"] = dt.dt.month
df["day"] = dt.dt.day
df["dayofweek"] = dt.dt.dayofweek
df["hour"] = dt.dt.hour
df["weekofyear"] = dt.dt.isocalendar().week.astype(int)
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(np.int8)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

In [ ]:
X = df.drop(columns=["is_click", "session_id"])
y = df["is_click"]

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
numeric_features = [
    "age_level",
    "user_depth",
    "city_development_index",
    "var_1",
    "hour",
    "weekday",
    "day",
    "month",
    "year",
    "is_weekend"
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
]

categorical_features = [
    "product",
    "product_category_1",
    "product_category_2",
    "user_group_id",
    "gender"
]

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [ ]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

In [ ]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", model)
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
proba = pipeline.predict_proba(X_valid)[:,1]

In [ ]:
auc = roc_auc_score(y_valid, proba)
print("ROC AUC:", auc)

In [ ]:
joblib.dump(pipeline, "ctr_model.pkl")

In [ ]:
model = joblib.load("ctr_model.pkl")

pred = model.predict_proba(df_test)[:,1]